In [1]:
from pathlib import Path  # Work with file and folder paths.
import os  # Read environment variables by name.
from dotenv import load_dotenv  # Load settings from our local .env file.
import psycopg  # Connect Python to PostgreSQL.

project_root = Path.cwd().parent  # Move from ingestion/ to the project root.
env_path = project_root / ".env"  # Locate the local settings file.
settings_loaded = load_dotenv(env_path)  # Load its values into this notebook process.

print("Project root:", project_root)  # Confirm the folder we are using.
print(".env found:", env_path.exists())  # Confirm the file exists.
print("Settings loaded:", settings_loaded)  # Confirm dotenv read settings.
print("Database name:", os.getenv("POSTGRES_DB"))  # Show a safe setting, not the password.

Project root: f:\Big Data\Retail_Lakehouse_Learning
.env found: True
Settings loaded: True
Database name: retail_lakehouse


In [2]:
with psycopg.connect(  # Open a connection to PostgreSQL.
    host="127.0.0.1",  # Docker exposed PostgreSQL on your computer.
    port=5433,  # Your computer's port; PostgreSQL uses 5432 inside Docker.
    dbname=os.environ["POSTGRES_DB"],  # Read the database name from .env.
    user=os.environ["POSTGRES_USER"],  # Read the database user from .env.
    password=os.environ["POSTGRES_PASSWORD"],  # Use the password without displaying it.
) as connection:  # Close the connection when this block finishes.
    with connection.cursor() as cursor:  # Create an object for sending SQL.
        cursor.execute("SELECT COUNT(*) FROM customers")  # Ask PostgreSQL to count customer rows.
        customer_count = cursor.fetchone()[0]  # Take the first value from the one-row result.
        print("Customers in PostgreSQL:", customer_count)  # Display the count only.

Customers in PostgreSQL: 750


In [3]:
with psycopg.connect(  # Open a new connection for this query.
    host="127.0.0.1",  # Connect through your computer's Docker port.
    port=5433,  # Use the published PostgreSQL port.
    dbname=os.environ["POSTGRES_DB"],  # Get the database name from .env.
    user=os.environ["POSTGRES_USER"],  # Get the username from .env.
    password=os.environ["POSTGRES_PASSWORD"],  # Get the password without printing it.
) as connection:  # Close the connection after the query.
    with connection.cursor() as cursor:  # Create a cursor to run SQL.
        cursor.execute(  # Ask for a small, predictable sample.
            "SELECT customer_id, city, loyalty_tier, join_date, active_flag "
            "FROM customers ORDER BY customer_id LIMIT 3"
        )
        sample_rows = cursor.fetchall()  # Bring the three result rows into Python.

print("Result container:", type(sample_rows).__name__)  # Usually a list.
for row in sample_rows:  # Visit each returned row.
    print(row)  # Each row is a tuple; dates and booleans have Python types.

Result container: list
('C00001', 'Lahore', 'Gold', datetime.date(2026, 5, 13), True)
('C00002', 'Karachi', 'Silver', datetime.date(2025, 8, 18), True)
('C00003', 'Karachi', 'Silver', datetime.date(2025, 4, 19), True)
